### Homework 8

#### Set up

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import json

from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from PIL import Image
from torchvision.datasets import ImageFolder

In [2]:


SEED = 42
NUM_WORKERS = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cpu")

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = ImageFolder('../00-data/data/train', transform=train_transform)
evaluation_dataset = ImageFolder('../00-data/data/test', transform=evaluation_transform)


loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,
)
evaluation_loader = DataLoader(
    evaluation_dataset,
    batch_size=20,
    shuffle=False,
    num_workers=0,
)

In [4]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

model = HairModel()
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

#### Question 1
Which loss matches the model above?


In [5]:
print('nn.BCEWithLogitsLoss()')
print(len(train_dataset), len(evaluation_dataset))

nn.BCEWithLogitsLoss()
800 201


#### Question 2
What is the total number of trainable parameters? Count biases as well and use sum(p.numel() for p in model.parameters()).

In [6]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

Total parameters: 20073473


#### Train metrics

Train for 10 epochs with the baseline transform. For every epoch, evaluate the held-out split with model.eval() and torch.no_grad(). Store these four series in a JSON file named history_baseline.json:

{
    "train_loss": [...],
    "train_accuracy": [...],
    "evaluation_loss": [...],
    "evaluation_accuracy": [...],
}

For accuracy, use (torch.sigmoid(logits) >= 0.5). Use the number of examples, not the number of batches, when averaging loss and accuracy.

In [7]:
num_epochs = 10
history = {'train_loss': [], 'train_accuracy': [], 'evaluation_loss': [], 'evaluation_accuracy': []}

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        labels = labels.unsqueeze(1).float()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        predicted = (torch.sigmoid(outputs) >= 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    train_loss = running_loss / total_train
    train_accuracy = correct_train / total_train
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_accuracy)
    
    model.eval() # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in evaluation_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            labels = labels.unsqueeze(1).float()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            predicted = (torch.sigmoid(outputs) >= 0.5).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss /= val_total
    val_acc = val_correct / val_total
    history['evaluation_loss'].append(val_loss)
    history['evaluation_accuracy'].append(val_acc)
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

output_file_name = 'history_baseline.json'
with open(output_file_name, 'w') as f:
    json.dump(history, f, indent=4)

print(f'Saved history to {output_file_name}')


Epoch 1/10
  Train Loss: 0.6385, Train Acc: 0.6450
  Val Loss: 0.6010, Val Acc: 0.6567
Epoch 2/10
  Train Loss: 0.5441, Train Acc: 0.7175
  Val Loss: 0.6003, Val Acc: 0.6517
Epoch 3/10
  Train Loss: 0.4955, Train Acc: 0.7550
  Val Loss: 0.6405, Val Acc: 0.6219
Epoch 4/10
  Train Loss: 0.4330, Train Acc: 0.7913
  Val Loss: 0.6274, Val Acc: 0.6915
Epoch 5/10
  Train Loss: 0.4048, Train Acc: 0.8087
  Val Loss: 0.6293, Val Acc: 0.6617
Epoch 6/10
  Train Loss: 0.3953, Train Acc: 0.8075
  Val Loss: 0.6332, Val Acc: 0.7114
Epoch 7/10
  Train Loss: 0.3437, Train Acc: 0.8363
  Val Loss: 0.6895, Val Acc: 0.6716
Epoch 8/10
  Train Loss: 0.2806, Train Acc: 0.8850
  Val Loss: 0.6497, Val Acc: 0.7214
Epoch 9/10
  Train Loss: 0.2236, Train Acc: 0.9100
  Val Loss: 0.7114, Val Acc: 0.7164
Epoch 10/10
  Train Loss: 0.1584, Train Acc: 0.9387
  Val Loss: 1.0212, Val Acc: 0.6368
Saved history to history_baseline.json


#### Question 3

Calculate the median of train_accuracy in history_baseline.json. Submit the value rounded to two decimal places. The grader accepts an absolute error of 0.05; the answer is not selected from a fixed option list.

In [8]:
with open('history_baseline.json', 'r') as f:
    history_baseline = json.load(f)
    train_accuracy_baseline = history_baseline['train_accuracy']
    median_train_accuracy_baseline = round(np.median(train_accuracy_baseline), 2)
    print(f'Median of train_accuracy_baseline: {median_train_accuracy_baseline}')

Median of train_accuracy_baseline: 0.81


#### Question 4

Calculate the population standard deviation of train_loss in history_baseline.json (for NumPy, use np.std(values, ddof=0)). Submit the value rounded to three decimal places. The grader accepts an absolute error of 0.02.

In [9]:
with open('history_baseline.json') as f:
    history_baseline = json.load(f)
    train_loss_baseline = history_baseline['train_loss']
    std_train_loss_baseline = round(np.std(train_loss_baseline, ddof=0), 3)
    print(f'Standard deviation of train loss: {std_train_loss_baseline}')

Standard deviation of train loss: 0.139


##### Data augmentation

Keep the same model and optimizer. Replace only the training transform.

Do not recreate the model or optimizer. Train for 10 additional epochs and write only those ten epochs to history_augmented.json. Continue using the unchanged, deterministic evaluation_dataset and evaluation_loader for every evaluation.

In [10]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset_augmented = ImageFolder('../00-data/data/train', transform=train_transform_augmented)

train_loader_augmented = DataLoader(
    train_dataset_augmented,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED+1),
)

In [11]:
num_epochs_augmented = 10
history_augmented = {'train_loss': [], 'train_accuracy': [], 'evaluation_loss': [], 'evaluation_accuracy': []}

for epoch in range(num_epochs_augmented):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in train_loader_augmented:
        inputs = inputs.to(device)
        labels = labels.to(device)
        labels = labels.unsqueeze(1).float()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        predicted = (torch.sigmoid(outputs) >= 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    train_loss = running_loss / total_train
    train_accuracy = correct_train / total_train
    history_augmented['train_loss'].append(train_loss)
    history_augmented['train_accuracy'].append(train_accuracy)
    
    model.eval() # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in evaluation_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            labels = labels.unsqueeze(1).float()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            predicted = (torch.sigmoid(outputs) >= 0.5).float()
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss /= val_total
    val_acc = val_correct / val_total
    history_augmented['evaluation_loss'].append(val_loss)
    history_augmented['evaluation_accuracy'].append(val_acc)
    
    print(f'Epoch {epoch+1}/{num_epochs_augmented}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

output_file_name = 'history_augmented.json'
with open(output_file_name, 'w') as f:
    json.dump(history_augmented, f, indent=4)

print(f'Saved history to {output_file_name}')


Epoch 1/10
  Train Loss: 0.6758, Train Acc: 0.6050
  Val Loss: 0.6552, Val Acc: 0.6716
Epoch 2/10
  Train Loss: 0.6129, Train Acc: 0.6475
  Val Loss: 0.6608, Val Acc: 0.6269
Epoch 3/10
  Train Loss: 0.5899, Train Acc: 0.6800
  Val Loss: 0.5787, Val Acc: 0.7114
Epoch 4/10
  Train Loss: 0.5923, Train Acc: 0.6750
  Val Loss: 0.5679, Val Acc: 0.7114
Epoch 5/10
  Train Loss: 0.5604, Train Acc: 0.7137
  Val Loss: 0.5680, Val Acc: 0.7214
Epoch 6/10
  Train Loss: 0.5567, Train Acc: 0.7087
  Val Loss: 0.5510, Val Acc: 0.7015
Epoch 7/10
  Train Loss: 0.5441, Train Acc: 0.7025
  Val Loss: 0.5288, Val Acc: 0.7413
Epoch 8/10
  Train Loss: 0.5329, Train Acc: 0.7212
  Val Loss: 0.5401, Val Acc: 0.7114
Epoch 9/10
  Train Loss: 0.5019, Train Acc: 0.7612
  Val Loss: 0.5632, Val Acc: 0.7363
Epoch 10/10
  Train Loss: 0.5312, Train Acc: 0.7262
  Val Loss: 0.5404, Val Acc: 0.7313
Saved history to history_augmented.json


#### Question 5

Calculate the mean of evaluation_loss in history_augmented.json. Submit the value rounded to three decimal places. The grader accepts an absolute error of 0.02.

In [12]:
with open('history_augmented.json', 'r') as fa:
    history_augmented = json.load(fa)
    evaluation_loss_augmented = history_augmented['evaluation_loss']
    mean_evaluation_loss_augmented = round(np.mean(evaluation_loss_augmented), 2)
    print(f'Mean of evaluation_loss_augmented: {mean_evaluation_loss_augmented}')

Mean of evaluation_loss_augmented: 0.58


#### Question 6

Calculate the mean of the last five values in evaluation_accuracy in history_augmented.json (epochs 6 through 10 of the augmented phase). Submit the value rounded to two decimal places. The grader accepts an absolute error of 0.05.

The tolerances cover minor CPU wheel differences, while the fixed seed, loader, split, architecture, and inference transform prevent the large uncontrolled variation from the 2025 exercise.

In [13]:
with open('history_augmented.json', 'r') as fa:
    history_augmented = json.load(fa)
    evaluation_accuracy_augmented = history_augmented['evaluation_accuracy']
    last_values = evaluation_accuracy_augmented[5:]
    mean_evaluation_accuracy_augmented = round(np.mean(last_values), 2)
    print(f'Mean of evaluation_accuracy_augmented: {mean_evaluation_accuracy_augmented}')

Mean of evaluation_accuracy_augmented: 0.72
